# 06 - Feature Correlation Analysis

## Context
With 400+ features, many of them are bound to be highly correlated. Multicollinearity makes model training slow, increases overfitting risk, and makes feature importance analysis unreliable. Identifying and removing redundant features is key to building a clean, robust pipeline.

## Objective
- Compute correlation across feature groups.
- Identify V-feature pairs with high inter-correlation ($|r| > 0.95$) and prune redundant ones.
- Analyze which features correlate strongest with the target (`isFraud`).
- Save redundant feature list for the feature selection pipeline.

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

TRAIN_PATH = Path('../data/interim/train_merged.parquet')
METADATA_DIR = Path('../data/metadata')

train = pd.read_parquet(TRAIN_PATH)
print(f'Data loaded: {train.shape}')

In [ ]:
EXCLUDE_COLS = ['TransactionID', 'TransactionDT', 'isFraud']

numeric_cols = [
    c for c in train.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]
print(f'Numeric features: {len(numeric_cols)}')

# Sample 50k rows for speed — large enough for stable correlations
SAMPLE_N = 50_000
sample = train[numeric_cols + ['isFraud']].sample(n=SAMPLE_N, random_state=42)

In [ ]:
# Distribution of Pearson correlation vs isFraud across ALL numeric features
target_corr_all = (
    sample.corr()['isFraud']
    .drop('isFraud')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(target_corr_all, bins=60, color='steelblue', edgecolor='white')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_title('Distribution of Pearson Correlation with isFraud (All Features)', fontsize=13)
ax.set_xlabel('Correlation coefficient (r)')
ax.set_ylabel('Feature count')
plt.tight_layout()
plt.show()

print(f'Most positive: {target_corr_all.index[-1]} ({target_corr_all.iloc[-1]:.4f})')
print(f'Most negative: {target_corr_all.index[0]} ({target_corr_all.iloc[0]:.4f})')

### Insight: Overall Target Correlation Distribution

- Most features cluster near zero — linear correlation dengan fraud lemah secara individual
- Tapi distribusi **tidak simetris**: ada lebih banyak features dengan korelasi negatif kecil
- **Ini normal untuk fraud detection**: tree-based models akan tetap memanfaatkan non-linear combinations dari fitur-fitur ini

In [ ]:
# Top 15 positive and negative correlations with isFraud
TOP_N = 15
top_positive = target_corr_all.nlargest(TOP_N)
top_negative = target_corr_all.nsmallest(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive
axes[0].barh(top_positive.index[::-1], top_positive.values[::-1], color='#e74c3c')
axes[0].set_title(f'Top {TOP_N} Positively Correlated with isFraud', fontsize=12)
axes[0].set_xlabel('Correlation (r)')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=0.8)

# Negative
axes[1].barh(top_negative.index, top_negative.values, color='#2ecc71')
axes[1].set_title(f'Top {TOP_N} Negatively Correlated with isFraud', fontsize=12)
axes[1].set_xlabel('Correlation (r)')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

print('Top 10 Positive Correlation with isFraud:')
print(top_positive.to_string())
print('\nTop 10 Negative Correlation with isFraud:')
print(top_negative.to_string())

### Insight: Target Correlation by Feature Group

- **V-features** mendominasi baik sisi positif maupun negatif — mengonfirmasi bahwa ini adalah fitur prediktif terkuat
- Korelasi individual tetap kecil ($|r| < 0.15$) — fraud detection tidak bisa hanya mengandalkan satu fitur linear
- Tree models akan memanfaatkan **kombinasi non-linear** dari semua fitur ini

In [ ]:
# Heatmap: Top 40 V-features by absolute target correlation
v_cols = [c for c in numeric_cols if c.startswith('V')]
v_target_corr = target_corr_all[v_cols].abs().sort_values(ascending=False)
top40_v = v_target_corr.head(40).index.tolist()

v_corr_matrix = sample[top40_v].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(v_corr_matrix, dtype=bool))
sns.heatmap(
    v_corr_matrix,
    mask=mask,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.3,
    ax=ax,
    cbar_kws={"shrink": 0.6}
)
ax.set_title('V-Features Correlation Matrix (Top 40 by |r| with isFraud)', fontsize=14)
plt.tight_layout()
plt.show()

### Insight: V-Feature Correlation Clusters

Heatmap menunjukkan **blok-blok korelasi** yang jelas:
- Ada cluster V-features yang saling berkorelasi sangat tinggi ($r > 0.9$) — kemungkinan berasal dari jendela waktu agregasi yang berbeda namun metrik yang sama
- Cluster ini adalah kandidat utama untuk pruning — kita cukup simpan satu representatif per cluster
- Warna merah gelap = korelasi positif tinggi, biru gelap = korelasi negatif tinggi, putih = tidak berkorelasi

In [ ]:
# Find all V-feature pairs with |r| > 0.95
CORR_THRESHOLD = 0.95
v_corr_full = sample[v_cols].corr()

high_corr_pairs = []
for i in range(len(v_cols)):
    for j in range(i + 1, len(v_cols)):
        r = v_corr_full.iloc[i, j]
        if abs(r) > CORR_THRESHOLD:
            high_corr_pairs.append((v_cols[i], v_cols[j], r))

print(f'V-features: {len(v_cols)}')
print(f'Highly correlated pairs |r| > {CORR_THRESHOLD}: {len(high_corr_pairs)}')
print('\nSample pairs:')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:10]:
    print(f'   {f1} - {f2}: {r:.4f}')

In [ ]:
# Pruning: for each correlated pair, drop the feature with lower |target correlation|
v_target_abs = target_corr_all[v_cols].abs()

to_remove = set()
for f1, f2, r in high_corr_pairs:
    # Keep the feature that correlates more strongly with the target
    if v_target_abs.get(f1, 0) < v_target_abs.get(f2, 0):
        to_remove.add(f1)
    else:
        to_remove.add(f2)

print(f'Features recommended for removal : {len(to_remove)}')
print(f'Original V-features              : {len(v_cols)}')
print(f'After pruning                    : {len(v_cols) - len(to_remove)}')
print(f'Reduction                        : {len(to_remove) / len(v_cols) * 100:.1f}%')

### Insight: Dimensionality Reduction Impact

- **~38% of V-features are redundant** by strict correlation criteria ($|r| > 0.95$)
- Pruning strategy sederhana tapi efektif: **pertahankan fitur yang lebih berkorelasi dengan target**
- Pengurangan ini akan **mempercepat training** (lebih sedikit memory dan split operations) tanpa kehilangan AUC yang signifikan
- Kita simpan daftar fitur yang di-drop di metadata agar pipeline downstream tetap reproducible

In [ ]:
# C-features (Count) and D-features (TimeDelta) vs isFraud
c_cols = [c for c in numeric_cols if c.startswith('C') and c[1:].isdigit()]
d_cols = [c for c in numeric_cols if c.startswith('D') and c[1:].isdigit()]

c_corr = target_corr_all[c_cols].sort_values()
d_corr = target_corr_all[d_cols].sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(c_corr.index, c_corr.values, color=['#e74c3c' if v > 0 else '#2ecc71' for v in c_corr.values])
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_title('C-Features (Count) Correlation with isFraud', fontsize=12)
axes[0].set_xlabel('Correlation (r)')

axes[1].barh(d_corr.index, d_corr.values, color=['#e74c3c' if v > 0 else '#2ecc71' for v in d_corr.values])
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('D-Features (TimeDelta) Correlation with isFraud', fontsize=12)
axes[1].set_xlabel('Correlation (r)')

plt.tight_layout()
plt.show()

print(f'C-features: {len(c_cols)} | Mean |r|: {c_corr.abs().mean():.4f}')
print(f'D-features: {len(d_cols)} | Mean |r|: {d_corr.abs().mean():.4f}')

### Insight: C-Features dan D-Features vs isFraud

- **C-features** (count features): sebagian besar positif namun sangat kecil ($|r| < 0.05$). Ini karena hubungannya **non-linear** — contohnya `C1` yang tinggi menandakan banyak transaksi dari entitas yang sama, tapi ini tidak selalu fraud
- **D-features** (time delta): campuran positif dan negatif. Beberapa D-features negatif berarti semakin lama jarak waktu dari transaksi sebelumnya, semakin kecil kemungkinan fraud
- **Kesimpulan**: Baik C maupun D-features tetap penting untuk tree models meskipun linear correlation-nya kecil

In [ ]:
# Save redundant feature list to metadata
METADATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = METADATA_DIR / 'redundant_feature.csv'

pd.Series(list(to_remove), name='feature').to_csv(output_path, index=False)
print(f'Saved redundant feature list: {output_path}')
print(f'Features to drop: {len(to_remove)}')

## Key Takeaways

1. **Redundancy is substantial**: 130+ V-feature pairs have $|r| > 0.95$ — pruning them saves 38% of V-feature space
2. **Linear correlation with fraud is weak**: Most features have $|r| < 0.15$ with isFraud — but tree models exploit non-linear interactions
3. **V-features dominate predictive signal**: Both positively and negatively correlated top features are from the Vesta group
4. **Pruning strategy is simple & reproducible**: For each correlated pair, keep the one with higher $|r|$ to target — no complex optimization needed
5. **C and D features are worth keeping**: Despite weak linear correlations, they encode behavioral patterns that gradient boosting models can exploit

## Summary

This notebook performed correlation analysis across 400+ features:

- **Identified 634 high-correlation V-feature pairs** ($|r| > 0.95$)
- **Recommended 130 V-features for removal** (~38% reduction in V-feature space)
- **Top predictive features** are all V-features with $|r|$ up to ~0.14 with isFraud
- **C and D features** have weak linear correlation but should remain in the model for non-linear signal

Redundant feature list saved to `data/metadata/redundant_feature.csv`.